<a href="https://colab.research.google.com/github/tonasthesecond/cs313-stock-analysis/blob/main/240069_project_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final project: Time-series data and application to stock markets {-}

In [ ]:
#@title imports
# standard library
import os
import glob
import re
import json
import subprocess
import threading
import time
from pathlib import Path
from itertools import product

# data / numeric
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# deep learning
import tensorflow as tf
from tensorflow.keras.layers import (
    Conv1D, MaxPooling1D, Flatten, Dense,
    LSTM, Dropout,
)

# ml utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight


In [ ]:
#@title setup
# Detects whether we're running in Colab or locally.
# In Colab: clones the repo if not present, sets BASE to the clone.
# Locally:  sets BASE to cwd (run notebook from repo root).

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    BASE = Path('/content/cs313-stock-analysis')
    if not BASE.exists():
        subprocess.run(
            ['git', 'clone',
             'https://github.com/tonasthesecond/cs313-stock-analysis',
             str(BASE)],
            check=True,
        )
else:
    BASE = Path('.').resolve()  # run from repo root

# ── paths ──────────────────────────────────────────────────────────────────
MODEL_DIR       = str(BASE / 'models')
NASDAQ_DATA_DIR = str(BASE / 'data' / 'nasdaq')
VN_DATA_DIR     = str(BASE / 'data' / 'vn')

for d in [MODEL_DIR, NASDAQ_DATA_DIR, VN_DATA_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'BASE:        {BASE}')
print(f'nasdaq data: {NASDAQ_DATA_DIR}')
print(f'vn data:     {VN_DATA_DIR}')
print(f'models:      {MODEL_DIR}')


In [4]:
#@title utilities

def split_data(X, y):
    """Chronological 64/16/20 train/val/test split. No shuffling."""
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
    X_train, X_val,  y_train, y_val  = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)
    return (np.array(X_train), np.array(X_val),  np.array(X_test),
            np.array(y_train), np.array(y_val),   np.array(y_test))


def normalize_windows(X, y, close_idx):
    """Per-window min-max normalization. y is normalized against its window's Close range."""
    X_norm = X.copy().astype(float)
    y_norm = y.copy().astype(float)
    mins   = np.zeros((len(X), X.shape[2]))
    maxs   = np.zeros((len(X), X.shape[2]))
    for i in range(len(X)):
        for f in range(X.shape[2]):
            mn, mx = X[i, :, f].min(), X[i, :, f].max()
            denom  = mx - mn if mx != mn else 1e-8
            X_norm[i, :, f] = (X[i, :, f] - mn) / denom
            mins[i, f], maxs[i, f] = mn, mx
        mn_c   = mins[i, close_idx]
        mx_c   = maxs[i, close_idx]
        denom_c = mx_c - mn_c if mx_c != mn_c else 1e-8
        y_norm[i] = (y[i] - mn_c) / denom_c
    return X_norm, y_norm, mins, maxs


def filter_flat_windows(X, y, close_idx, min_range_pct=0.001):
    """Drop windows where Close barely moves — causes normalization explosion."""
    X, y = np.array(X), np.array(y)
    keep = []
    for i in range(len(X)):
        closes     = X[i, :, close_idx]
        mn, mx     = closes.min(), closes.max()
        mean       = closes.mean()
        if mean > 0 and (mx - mn) / mean >= min_range_pct:
            keep.append(i)
    keep = np.array(keep, dtype=int)
    return X[keep], y[keep]


def window_data(df, features, close_idx, window_size, forecast_day=1, forecast_days=None):
    """
    Slide a window over df and build (X, y) arrays.
    forecast_day:  predict a single day N steps ahead (default 1 = next day)
    forecast_days: predict K consecutive days ahead (overrides forecast_day)
    """
    arr     = df[features].values
    horizon = forecast_days if forecast_days else forecast_day
    X, y    = [], []
    for i in range(len(arr) - window_size - horizon):
        window = arr[i : i + window_size]
        if forecast_days:
            label = arr[i + window_size : i + window_size + forecast_days, close_idx]
        else:
            label = [arr[i + window_size + forecast_day - 1, close_idx]]
        X.append(window)
        y.append(label)
    return np.array(X), np.array(y)


def prepare_splits(X, y, close_idx):
    """Split and normalize pre-built windows. Returns normalized splits + raw test labels."""
    X_tr, X_v, X_te, y_tr, y_v, y_te = split_data(X, y)
    X_tr_n, y_tr_n, _,    _           = normalize_windows(X_tr, y_tr, close_idx)
    X_v_n,  y_v_n,  _,    _           = normalize_windows(X_v,  y_v,  close_idx)
    X_te_n, _,      mins, maxs         = normalize_windows(X_te, y_te, close_idx)
    print(f'train: {len(X_tr)}  val: {len(X_v)}  test: {len(X_te)}')
    return X_tr_n, y_tr_n, X_v_n, y_v_n, X_te_n, y_te, mins, maxs


def prepare_data(data, features, close_idx, window_size, forecast_day=1, forecast_days=None):
    """Window → filter flat → split → normalize. Use for datasets with flat-price artifacts."""
    X, y = window_data(data, features, close_idx, window_size, forecast_day, forecast_days)
    X, y = filter_flat_windows(X, y, close_idx)
    print(f'windows after filter: {len(X)}')
    return prepare_splits(X, y, close_idx)


def train_or_load_model(model_path, builder, window_size, n_features, n_outputs,
                         X_tr_n, y_tr_n, X_v_n, y_v_n,
                         epochs=30, batch_size=64):
    """Load model from disk if it exists, otherwise train with builder and save."""
    if os.path.exists(model_path):
        print(f'loading {model_path}')
        return tf.keras.models.load_model(model_path), None
    model   = builder(window_size, n_features, n_outputs)
    history = model.fit(
        X_tr_n, y_tr_n,
        validation_data=(X_v_n, y_v_n),
        epochs=epochs, batch_size=batch_size,
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
    )
    model.save(model_path)
    print(f'saved to {model_path}')
    return model, history


def denorm_predictions(y_pred_norm, mins, maxs, close_idx):
    """Denormalize model output using per-window Close min/max."""
    return np.array([
        y_pred_norm[i] * (maxs[i, close_idx] - mins[i, close_idx]) + mins[i, close_idx]
        for i in range(len(y_pred_norm))
    ])


def evaluate_and_plot(model, X_te_n, y_te, mins, maxs, close_idx, title, currency='$'):
    """Predict, denormalize, plot predicted vs actual, and print RMSE/MAE per output day."""
    y_pred_norm   = model.predict(X_te_n, verbose=0)
    y_pred_denorm = denorm_predictions(y_pred_norm, mins, maxs, close_idx)
    n_outputs     = y_pred_denorm.shape[1]

    plt.figure(figsize=(16, 8), dpi=150)
    for d in range(n_outputs):
        plt.plot(y_pred_denorm[:, d], label=f'predicted day +{d+1}', alpha=0.7)
    plt.plot(y_te[:, 0], label='actual', color='black', linewidth=1.5)
    plt.title(title, fontsize=14)
    plt.xlabel('time (days)')
    plt.ylabel(f'close price ({currency})')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

    print(f"\n{'day':<8} {'RMSE':>10} {'MAE':>10}")
    print('-' * 30)
    for d in range(n_outputs):
        rmse = np.sqrt(mean_squared_error(y_te[:, d], y_pred_denorm[:, d]))
        mae  = mean_absolute_error(y_te[:, d], y_pred_denorm[:, d])
        print(f'+{d+1:<7} {rmse:>10.4f} {mae:>10.4f}')


def add_technical_indicators(df):
    """Append SMA, RSI, MACD, and rolling volatility columns to a price DataFrame."""
    df    = df.copy()
    close = df['Close']

    df['SMA5']  = close.rolling(5).mean()
    df['SMA20'] = close.rolling(20).mean()

    delta       = close.diff()
    gain        = delta.clip(lower=0).rolling(14).mean()
    loss        = (-delta.clip(upper=0)).rolling(14).mean()
    rs          = gain / loss.replace(0, 1e-8)
    df['RSI14'] = 100 - (100 / (1 + rs))

    ema12             = close.ewm(span=12, adjust=False).mean()
    ema26             = close.ewm(span=26, adjust=False).mean()
    df['MACD']        = ema12 - ema26
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

    df['Volatility'] = close.pct_change().rolling(10).std()

    return df.dropna().reset_index(drop=True)

# 1: Nasdaq stock price prediction

In [ ]:
#@title nasdaq settings
# ── features ──────────────────────────────────────────────────────────────
# All six OHLCAV columns: intraday range (High/Low), liquidity (Volume),
# and split-adjusted history (Adjusted Close).
FEATURES           = ['Open', 'High', 'Low', 'Close', 'Adjusted Close', 'Volume']
CLOSE_IDX          = FEATURES.index('Close')

# ── hyperparameters ────────────────────────────────────────────────────────
NASDAQ_WINDOW_SIZE = 30   # ~1.5 calendar months of lookback
FORECAST_DAY       = 3    # task 1.2: predict the price N days ahead
FORECAST_DAYS      = 3    # task 1.3: predict the next N consecutive days


In [ ]:
#@title load nasdaq data
# Loads one company CSV sorted chronologically as a representative sample.
nasdaq_files    = sorted(glob.glob(f'{NASDAQ_DATA_DIR}/*.csv'))
data            = pd.read_csv(nasdaq_files[0]).sort_values('Date').reset_index(drop=True)
nasdaq_filename = Path(nasdaq_files[0]).stem
print(f'{nasdaq_filename} — {len(data)} rows  |  columns: {list(data.columns)}')
data[FEATURES].head()


In [ ]:
#@title nasdaq model
# CNN regressor: three Conv1D stages at increasing depth, then a Dense head.
# Convolutional kernels detect local temporal patterns (momentum, consolidation)
# at multiple time scales. No recurrent layers needed for the smooth Nasdaq series.
def build_nasdaq_model(window_size, n_features, n_outputs=1):
    inputs = tf.keras.Input(shape=(window_size, n_features))
    x = Conv1D(64,  kernel_size=3, activation='relu', padding='same')(inputs)
    x = MaxPooling1D(2)(x)
    x = Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = MaxPooling1D(2)(x)
    x = Conv1D(64,  kernel_size=3, activation='relu', padding='same')(x)
    x = MaxPooling1D(2)(x)
    x = Flatten()(x)
    x = Dense(100, activation='relu')(x)
    outputs = Dense(n_outputs)(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-2), loss='mse', metrics=['mse'])
    return model


## 1.1: next-day forecast

In [ ]:
# Task 1.1 — next-day Close prediction.
# window_data builds (X, y) pairs; prepare_splits does chron 64/16/20 + normalisation.
X, y = window_data(data, FEATURES, CLOSE_IDX, NASDAQ_WINDOW_SIZE)
X_tr_n, y_tr_n, X_v_n, y_v_n, X_te_n, y_te, mins, maxs = prepare_splits(X, y, CLOSE_IDX)
model_1_1, _ = train_or_load_model(
    f'{MODEL_DIR}/task1_1_{nasdaq_filename}.keras', build_nasdaq_model,
    NASDAQ_WINDOW_SIZE, len(FEATURES), 1,
    X_tr_n, y_tr_n, X_v_n, y_v_n,
)
evaluate_and_plot(model_1_1, X_te_n, y_te, mins, maxs, CLOSE_IDX,
                  'task 1.1 — next-day forecast')


## 1.2: nth-day forecast

In [ ]:
# Task 1.2 — predict FORECAST_DAY days ahead (set in nasdaq settings).
# Same architecture; only the label target index shifts.
X, y = window_data(data, FEATURES, CLOSE_IDX, NASDAQ_WINDOW_SIZE, forecast_day=FORECAST_DAY)
X_tr_n, y_tr_n, X_v_n, y_v_n, X_te_n, y_te, mins, maxs = prepare_splits(X, y, CLOSE_IDX)
model_1_2, _ = train_or_load_model(
    f'{MODEL_DIR}/task1_2_day{FORECAST_DAY}_{nasdaq_filename}.keras', build_nasdaq_model,
    NASDAQ_WINDOW_SIZE, len(FEATURES), 1,
    X_tr_n, y_tr_n, X_v_n, y_v_n,
)
evaluate_and_plot(model_1_2, X_te_n, y_te, mins, maxs, CLOSE_IDX,
                  f'task 1.2 — day +{FORECAST_DAY} forecast')


## 1.3: k-days forecast

In [ ]:
# Task 1.3 — predict FORECAST_DAYS consecutive closes (set in nasdaq settings).
# Output layer width = FORECAST_DAYS; MSE averages across all output steps.
X, y = window_data(data, FEATURES, CLOSE_IDX, NASDAQ_WINDOW_SIZE, forecast_days=FORECAST_DAYS)
X_tr_n, y_tr_n, X_v_n, y_v_n, X_te_n, y_te, mins, maxs = prepare_splits(X, y, CLOSE_IDX)
model_1_3, _ = train_or_load_model(
    f'{MODEL_DIR}/task1_3_{FORECAST_DAYS}days_{nasdaq_filename}.keras', build_nasdaq_model,
    NASDAQ_WINDOW_SIZE, len(FEATURES), FORECAST_DAYS,
    X_tr_n, y_tr_n, X_v_n, y_v_n,
)
evaluate_and_plot(model_1_3, X_te_n, y_te, mins, maxs, CLOSE_IDX,
                  f'task 1.3 — {FORECAST_DAYS}-day consecutive forecast')


# 2: Vietnam stock price prediction

In [ ]:
#@title vietnam settings
# ── features ──────────────────────────────────────────────────────────────
# OHLCV + 6 technical indicators appended by add_technical_indicators().
# SMA5/20: trend direction | RSI14: momentum saturation
# MACD/signal: momentum vs medium-term trend | Volatility: risk regime
VN_FEATURES  = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'SMA5', 'SMA20', 'RSI14', 'MACD', 'MACD_signal', 'Volatility',
]
VN_CLOSE_IDX = VN_FEATURES.index('Close')

# ── hyperparameters ────────────────────────────────────────────────────────
# VN_WINDOW_SIZE=20 was selected by the architecture x window sweep (section 2.a).
# Shorter than Nasdaq: Vietnamese stocks are more volatile and a longer lookback
# risks spanning market-regime boundaries.
VN_WINDOW_SIZE   = 20
VN_FORECAST_DAY  = 3    # task 2.2: predict N days ahead
VN_FORECAST_DAYS = 3    # task 2.3: predict N consecutive days


In [ ]:
#@title load and enrich vietnam data
# Loads one Vietnam CSV and appends technical indicator columns.
# Rows covering the indicator warm-up period (~26 rows for EMA26) are dropped.
vn_files    = sorted(glob.glob(f'{VN_DATA_DIR}/*.csv'))
data_vn_raw = pd.read_csv(vn_files[0], index_col=0).sort_values('TradingDate').reset_index(drop=True)
data_vn     = add_technical_indicators(data_vn_raw)
vn_filename = Path(vn_files[0]).stem
print(f'{vn_filename} — {len(data_vn)} rows after warmup  (was {len(data_vn_raw)})')
data_vn[VN_FEATURES].head()


## 2.a: architecture trials

In [ ]:
#@title architecture definitions (v0-v4)
# Five architectures for the benchmark/sweep.
# v0: pure LSTM | v1: shallow CNN+LSTM | v2: deep CNN+LSTM (winner)
# v3: CNN+stacked LSTM | v4: CNN+LSTM+attention
def build_vn_model_v0(window_size, n_features, n_outputs=1):
    """pure sequential recurrent: 2 layers + dropout."""
    inputs = tf.keras.Input(shape=(window_size, n_features))
    x = LSTM(128, return_sequences=True)(inputs)
    x = Dropout(0.2)(x)
    x = LSTM(64,  return_sequences=False)(x)
    x = Dropout(0.2)(x)
    x = Dense(32, activation='relu')(x)
    outputs = Dense(n_outputs)(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(3e-4), loss='mse', metrics=['mse'])
    return model


def build_vn_model_v1(window_size, n_features, n_outputs=1):
    """CNN + recurrent: baseline hybrid."""
    inputs = tf.keras.Input(shape=(window_size, n_features))
    x = Conv1D(64,  kernel_size=3, activation='relu', padding='same')(inputs)
    x = MaxPooling1D(2)(x)
    x = Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = MaxPooling1D(2)(x)
    x = LSTM(64, return_sequences=False)(x)
    x = Dropout(0.2)(x)
    x = Dense(32, activation='relu')(x)
    outputs = Dense(n_outputs)(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(3e-4), loss='mse', metrics=['mse'])
    return model


def build_vn_model_v2(window_size, n_features, n_outputs=1):
    """deeper CNN + larger recurrent: best from sweep (window=20)."""
    inputs = tf.keras.Input(shape=(window_size, n_features))
    x = Conv1D(64,  kernel_size=3, activation='relu', padding='same')(inputs)
    x = Conv1D(64,  kernel_size=3, activation='relu', padding='same')(x)
    x = MaxPooling1D(2)(x)
    x = Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = MaxPooling1D(2)(x)
    x = LSTM(128, return_sequences=False)(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)
    outputs = Dense(n_outputs)(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(3e-4), loss='mse', metrics=['mse'])
    return model


def build_vn_model_v3(window_size, n_features, n_outputs=1):
    """CNN + stacked recurrent layers."""
    inputs = tf.keras.Input(shape=(window_size, n_features))
    x = Conv1D(64,  kernel_size=3, activation='relu', padding='same')(inputs)
    x = MaxPooling1D(2)(x)
    x = Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = MaxPooling1D(2)(x)
    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(0.2)(x)
    x = LSTM(32, return_sequences=False)(x)
    x = Dropout(0.2)(x)
    x = Dense(32, activation='relu')(x)
    outputs = Dense(n_outputs)(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(3e-4), loss='mse', metrics=['mse'])
    return model


def build_vn_model_v4(window_size, n_features, n_outputs=1):
    """CNN + recurrent + temporal attention."""
    inputs = tf.keras.Input(shape=(window_size, n_features))
    x = Conv1D(64,  kernel_size=3, activation='relu', padding='same')(inputs)
    x = MaxPooling1D(2)(x)
    x = Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = MaxPooling1D(2)(x)
    x = LSTM(64, return_sequences=True)(x)
    x = Dropout(0.2)(x)
    attn = tf.keras.layers.Dense(1, activation='tanh')(x)
    attn = tf.keras.layers.Flatten()(attn)
    attn = tf.keras.layers.Activation('softmax')(attn)
    attn = tf.keras.layers.RepeatVector(64)(attn)
    attn = tf.keras.layers.Permute([2, 1])(attn)
    x    = tf.keras.layers.Multiply()([x, attn])
    x    = tf.keras.layers.Lambda(lambda t: tf.reduce_sum(t, axis=1))(x)
    x    = Dense(32, activation='relu')(x)
    outputs = Dense(n_outputs)(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(3e-4), loss='mse', metrics=['mse'])
    return model

In [ ]:
#@title architecture benchmark (window=30)
# Trains (or loads) all five architectures at window=30 and compares RMSE/MAE.
BENCHMARK_RESULTS = {}
trial_builders    = {
    'v0': build_vn_model_v0,
    'v1': build_vn_model_v1,
    'v2': build_vn_model_v2,
    'v3': build_vn_model_v3,
    'v4': build_vn_model_v4,
}

X_tr_n, y_tr_n, X_v_n, y_v_n, X_te_n, y_te, mins, maxs = prepare_data(
    data_vn, VN_FEATURES, VN_CLOSE_IDX, 30
)

for name, builder in trial_builders.items():
    path  = f'{MODEL_DIR}/trial_arch_{name}.keras'
    model, _ = train_or_load_model(path, builder, 30, len(VN_FEATURES), 1,
                                    X_tr_n, y_tr_n, X_v_n, y_v_n, epochs=30, batch_size=64)
    y_pred = denorm_predictions(model.predict(X_te_n, verbose=0), mins, maxs, VN_CLOSE_IDX)
    rmse   = np.sqrt(mean_squared_error(y_te[:, 0], y_pred[:, 0]))
    mae    = mean_absolute_error(y_te[:, 0], y_pred[:, 0])
    BENCHMARK_RESULTS[name] = {'rmse': rmse, 'mae': mae}

print(f"\n{'arch':<8} {'RMSE':>10} {'MAE':>10}")
print('-' * 30)
for name, s in sorted(BENCHMARK_RESULTS.items(), key=lambda x: x[1]['rmse']):
    print(f"{name:<8} {s['rmse']:>10.4f} {s['mae']:>10.4f}")

In [ ]:
#@title window x architecture sweep
# Sweeps v0/v1/v2 over window sizes [10, 20, 30, 60] — 12 configs total.
# Models are cached; re-runs skip training.
SWEEP_RESULTS  = {}
sweep_builders = {'v0': build_vn_model_v0, 'v1': build_vn_model_v1, 'v2': build_vn_model_v2}
window_sizes   = [10, 20, 30, 60]

for ws, (arch, builder) in product(window_sizes, sweep_builders.items()):
    run  = f'{arch}_w{ws}'
    path = f'{MODEL_DIR}/trial_sweep_{run}.keras'
    X_tr_n, y_tr_n, X_v_n, y_v_n, X_te_n, y_te, mins, maxs = prepare_data(
        data_vn, VN_FEATURES, VN_CLOSE_IDX, ws
    )
    model, _ = train_or_load_model(path, builder, ws, len(VN_FEATURES), 1,
                                    X_tr_n, y_tr_n, X_v_n, y_v_n, epochs=30, batch_size=64)
    y_pred = denorm_predictions(model.predict(X_te_n, verbose=0), mins, maxs, VN_CLOSE_IDX)
    rmse   = np.sqrt(mean_squared_error(y_te[:, 0], y_pred[:, 0]))
    mae    = mean_absolute_error(y_te[:, 0], y_pred[:, 0])
    SWEEP_RESULTS[run] = {'rmse': rmse, 'mae': mae, 'window': ws, 'arch': arch}

print(f"\n{'run':<16} {'window':>8} {'RMSE':>10} {'MAE':>10}")
print('-' * 46)
for run, s in sorted(SWEEP_RESULTS.items(), key=lambda x: x[1]['rmse']):
    print(f"{run:<16} {s['window']:>8} {s['rmse']:>10.4f} {s['mae']:>10.4f}")

In [ ]:
#@title plot sweep results
# Bar chart of RMSE and MAE for all (architecture, window) combinations.
archs = list(sweep_builders.keys())
x     = np.arange(len(archs))
width = 0.2

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
for ax, metric in zip(axes, ['rmse', 'mae']):
    for i, ws in enumerate(window_sizes):
        values = [SWEEP_RESULTS[f'{arch}_w{ws}'][metric] for arch in archs]
        bars   = ax.bar(x + i * width, values, width, label=f'window={ws}')
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                    f'{val:.0f}', ha='center', va='bottom', fontsize=7)
    ax.set_title(metric.upper())
    ax.set_xticks(x + width)
    ax.set_xticklabels(archs)
    ax.set_ylabel('VND')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)
fig.suptitle('sweep: architecture x window size', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
#@title chosen model — v2 at window=20
# v2 (deeper paired CNN + LSTM128) at window=20 won both benchmarks.
# VN_WINDOW_SIZE is already set to 20 in vietnam settings above.
VN_MODEL_BUILDER = build_vn_model_v2
print(f'model: build_vn_model_v2  |  window: {VN_WINDOW_SIZE}')


## 2.1: next-day forecast

In [ ]:
# Task 2.1 — next-day Close prediction with 11-feature Vietnam data.
X_tr_n, y_tr_n, X_v_n, y_v_n, X_te_n, y_te, mins, maxs = prepare_data(
    data_vn, VN_FEATURES, VN_CLOSE_IDX, VN_WINDOW_SIZE
)
model_2_1, _ = train_or_load_model(
    f'{MODEL_DIR}/task2_1_{vn_filename}.keras', VN_MODEL_BUILDER,
    VN_WINDOW_SIZE, len(VN_FEATURES), 1,
    X_tr_n, y_tr_n, X_v_n, y_v_n,
)
evaluate_and_plot(model_2_1, X_te_n, y_te, mins, maxs, VN_CLOSE_IDX,
                  'task 2.1 — next-day forecast', currency='VND')


## 2.2: nth-day forecast

In [ ]:
# Task 2.2 — predict VN_FORECAST_DAY days ahead (set in vietnam settings).
X_tr_n, y_tr_n, X_v_n, y_v_n, X_te_n, y_te, mins, maxs = prepare_data(
    data_vn, VN_FEATURES, VN_CLOSE_IDX, VN_WINDOW_SIZE, forecast_day=VN_FORECAST_DAY
)
model_2_2, _ = train_or_load_model(
    f'{MODEL_DIR}/task2_2_day{VN_FORECAST_DAY}_{vn_filename}.keras', VN_MODEL_BUILDER,
    VN_WINDOW_SIZE, len(VN_FEATURES), 1,
    X_tr_n, y_tr_n, X_v_n, y_v_n,
)
evaluate_and_plot(model_2_2, X_te_n, y_te, mins, maxs, VN_CLOSE_IDX,
                  f'task 2.2 — day +{VN_FORECAST_DAY} forecast', currency='VND')


## 2.3: k-days forecast

In [ ]:
# Task 2.3 — predict VN_FORECAST_DAYS consecutive closes (set in vietnam settings).
X_tr_n, y_tr_n, X_v_n, y_v_n, X_te_n, y_te, mins, maxs = prepare_data(
    data_vn, VN_FEATURES, VN_CLOSE_IDX, VN_WINDOW_SIZE, forecast_days=VN_FORECAST_DAYS
)
model_2_3, _ = train_or_load_model(
    f'{MODEL_DIR}/task2_3_{VN_FORECAST_DAYS}days_{vn_filename}.keras', VN_MODEL_BUILDER,
    VN_WINDOW_SIZE, len(VN_FEATURES), VN_FORECAST_DAYS,
    X_tr_n, y_tr_n, X_v_n, y_v_n,
)
evaluate_and_plot(model_2_3, X_te_n, y_te, mins, maxs, VN_CLOSE_IDX,
                  f'task 2.3 — {VN_FORECAST_DAYS}-day consecutive forecast', currency='VND')


# 3: trading signal identification

In [ ]:
#@title signal utilities
# make_signal_labels: labels each timestep as buy/sell if the current Close is near
# the bottom/top percentile of a combined [past, future] window.
# build_signal_model: LSTM binary classifier, output in [0,1].
# train_or_load_signal_model: balanced class weights, early stopping.
def make_signal_labels(df, feature_cols, close_idx, past, future, mode='buy', threshold=0.2):
    """
    Label each timestep as a buy or sell signal.
    mode='buy':  1 if price is near the bottom of a past+future window.
    mode='sell': 1 if price is near the top of a past+future window.
    """
    closes = df[feature_cols[close_idx]].values
    X_data, y_data = [], []
    for i in range(past, len(df) - future - 1):
        window_features    = df[feature_cols].iloc[i - past : i].values
        full_window_closes = closes[i - past : i + future + 1]
        present_close      = closes[i]
        if mode == 'buy':
            label = 1 if present_close <= np.percentile(full_window_closes, threshold * 100) else 0
        else:
            label = 1 if present_close >= np.percentile(full_window_closes, (1 - threshold) * 100) else 0
        X_data.append(window_features)
        y_data.append(label)
    return np.array(X_data), np.array(y_data)


def build_signal_model(window_size, n_features):
    """Binary classifier for trading signal identification."""
    inputs = tf.keras.Input(shape=(window_size, n_features))
    x = LSTM(128, return_sequences=True)(inputs)
    x = Dropout(0.2)(x)
    x = LSTM(64,  return_sequences=False)(x)
    x = Dropout(0.2)(x)
    x = Dense(32, activation='relu')(x)
    outputs = Dense(1, activation='sigmoid')(x)
    model = tf.keras.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model


def train_or_load_signal_model(model_path, window_size, n_features,
                                X_tr, y_tr, X_v, y_v,
                                epochs=20, batch_size=256):
    """Load classifier if exists, otherwise train with class balancing and save."""
    if os.path.exists(model_path):
        print(f'loading {model_path}')
        return tf.keras.models.load_model(model_path), None
    classes      = np.unique(y_tr)
    weights      = compute_class_weight('balanced', classes=classes, y=y_tr)
    class_weight = dict(zip(classes, weights))
    print(f'class weights: {class_weight}')
    model   = build_signal_model(window_size, n_features)
    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_v, y_v),
        epochs=epochs, batch_size=batch_size,
        class_weight=class_weight,
        callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
    )
    model.save(model_path)
    return model, history

In [ ]:
#@title signal settings
# past:   days of history the classifier sees as input.
# future: days ahead used only for label generation (not visible at inference).
SIGNAL_WINDOW_PAST   = 10
SIGNAL_WINDOW_FUTURE = 10


## 3.1: buy signal

In [ ]:
# Task 3.1 — buy signal classifier.
# Label=1 when Close <= 20th percentile of the [past, future] window (local trough).
# Class balancing handles the ~20% positive rate.
X_buy, y_buy        = make_signal_labels(data_vn, VN_FEATURES, VN_CLOSE_IDX,
                                          SIGNAL_WINDOW_PAST, SIGNAL_WINDOW_FUTURE, mode='buy')
X_buy_norm, _, _, _ = normalize_windows(X_buy, y_buy.reshape(-1, 1), VN_CLOSE_IDX)
X_buy_tr, X_buy_v, X_buy_te, y_buy_tr, y_buy_v, y_buy_te = split_data(X_buy_norm, y_buy)

print(f'class balance — train: {y_buy_tr.mean():.3f}  test: {y_buy_te.mean():.3f}')

model_3_1, _ = train_or_load_signal_model(
    f'{MODEL_DIR}/task3_1_buy_{vn_filename}.keras',
    SIGNAL_WINDOW_PAST, len(VN_FEATURES),
    X_buy_tr, y_buy_tr, X_buy_v, y_buy_v,
)

y_pred_buy = (model_3_1.predict(X_buy_te, verbose=0) > 0.5).astype(int).flatten()
print(classification_report(y_buy_te, y_pred_buy, target_names=['hold', 'buy']))


## 3.2: sell signal

In [ ]:
# Task 3.2 — sell signal classifier.
# Label=1 when Close >= 80th percentile of the [past, future] window (local peak).
# Trained independently — entry and exit patterns are structurally different.
X_sell, y_sell          = make_signal_labels(data_vn, VN_FEATURES, VN_CLOSE_IDX,
                                              SIGNAL_WINDOW_PAST, SIGNAL_WINDOW_FUTURE, mode='sell')
X_sell_norm, _, _, _    = normalize_windows(X_sell, y_sell.reshape(-1, 1), VN_CLOSE_IDX)
X_sell_tr, X_sell_v, X_sell_te, y_sell_tr, y_sell_v, y_sell_te = split_data(X_sell_norm, y_sell)

print(f'class balance — train: {y_sell_tr.mean():.3f}  test: {y_sell_te.mean():.3f}')

model_3_2, _ = train_or_load_signal_model(
    f'{MODEL_DIR}/task3_2_sell_{vn_filename}.keras',
    SIGNAL_WINDOW_PAST, len(VN_FEATURES),
    X_sell_tr, y_sell_tr, X_sell_v, y_sell_v,
)

y_pred_sell = (model_3_2.predict(X_sell_te, verbose=0) > 0.5).astype(int).flatten()
print(classification_report(y_sell_te, y_pred_sell, target_names=['hold', 'sell']))


# 4: Profitable stock selection, risk management and portfolio composition

In [ ]:
#@title task 4 settings
# ── company filter ─────────────────────────────────────────────────────────
# 160 raw rows ≈ 120 usable points after indicator warm-up (26 rows) + window buffer.
MIN_ROWS_RAW    = 160

# ── portfolio params ───────────────────────────────────────────────────────
# Companies above RISK_PERCENTILE on (volatility + |max_drawdown|) are excluded.
RISK_PERCENTILE = 70

# How many top companies to display in the profitability ranking.
TOP_N           = 10


In [ ]:
#@title stage 1 — company filter
# Filters the full Vietnam universe to companies with sufficient data and liquidity.
vn_all_files       = sorted(glob.glob(f'{VN_DATA_DIR}/*.csv'))
filtered_companies = []

for path in vn_all_files:
    ticker = Path(path).stem
    raw    = pd.read_csv(path, index_col=0)
    if 'TradingDate' not in raw.columns:
        continue
    raw      = raw.sort_values('TradingDate').reset_index(drop=True)
    required = ['Open', 'High', 'Low', 'Close', 'Volume']
    if not all(c in raw.columns for c in required):
        continue
    if len(raw) < MIN_ROWS_RAW:
        continue
    if raw[required].isnull().any().any():
        continue
    if raw['Volume'].mean() < 1:
        continue
    filtered_companies.append({'ticker': ticker, 'path': path})

print(f'passed filter: {len(filtered_companies)} / {len(vn_all_files)} companies')


In [ ]:
#@title per-company training loop
# Trains (or loads) a v2 CNN-LSTM model per company. Stores predicted return and metrics.
COMPANY_RESULTS = {}

for entry in filtered_companies:
    ticker, path = entry['ticker'], entry['path']
    raw  = pd.read_csv(path, index_col=0).sort_values('TradingDate').reset_index(drop=True)
    data = add_technical_indicators(raw)
    if len(data) < MIN_ROWS_RAW:
        continue
    try:
        X_tr_n, y_tr_n, X_v_n, y_v_n, X_te_n, y_te, mins, maxs = prepare_data(
            data, VN_FEATURES, VN_CLOSE_IDX, VN_WINDOW_SIZE
        )
    except Exception as e:
        print(f'[skip] {ticker}: {e}')
        continue
    model_path = f'{MODEL_DIR}/task4_{ticker}.keras'
    model, _   = train_or_load_model(
        model_path, build_vn_model_v2,
        VN_WINDOW_SIZE, len(VN_FEATURES), 1,
        X_tr_n, y_tr_n, X_v_n, y_v_n,
    )
    y_pred         = denorm_predictions(model.predict(X_te_n, verbose=0), mins, maxs, VN_CLOSE_IDX)
    last_actual    = float(y_te[-1, 0])
    last_predicted = float(y_pred[-1, 0])
    COMPANY_RESULTS[ticker] = {
        'model_path':       model_path,
        'rmse':             float(np.sqrt(mean_squared_error(y_te[:, 0], y_pred[:, 0]))),
        'mae':              float(mean_absolute_error(y_te[:, 0], y_pred[:, 0])),
        'last_actual':      last_actual,
        'last_predicted':   last_predicted,
        'predicted_return': (last_predicted - last_actual) / last_actual if last_actual > 0 else 0.0,
    }

print(f'trained/loaded: {len(COMPANY_RESULTS)} companies')


## 4.1: profitable stock selection

In [ ]:
#@title 4.1 — rank by predicted return
# Sort all companies by predicted_return = (predicted - actual) / actual.
ranked = sorted(COMPANY_RESULTS.items(), key=lambda x: x[1]['predicted_return'], reverse=True)

print(f"{'ticker':<30} {'predicted_return':>18} {'last_actual':>14} {'last_predicted':>16}")
print('-' * 80)
for ticker, r in ranked[:TOP_N]:
    print(f"{ticker:<30} {r['predicted_return']:>17.4%} {r['last_actual']:>14.0f} {r['last_predicted']:>16.0f}")


In [ ]:
#@title 4.1 — select profitable candidates
# Keep only companies with positive predicted return as portfolio candidates.
profitable_tickers = [ticker for ticker, r in ranked if r['predicted_return'] > 0]
print(f'positive predicted return: {len(profitable_tickers)} companies')
print(f'top {TOP_N}: {[t for t, _ in ranked[:TOP_N]]}')


## 4.2: risk management

In [ ]:
#@title 4.2 — compute risk scores
# risk_score = volatility + |max_drawdown|.
# Volatility: day-to-day noise. Max drawdown: worst peak-to-trough crash.
for entry in filtered_companies:
    ticker = entry['ticker']
    if ticker not in COMPANY_RESULTS:
        continue
    raw    = pd.read_csv(entry['path'], index_col=0).sort_values('TradingDate').reset_index(drop=True)
    closes = raw['Close'].dropna().values

    daily_returns = np.diff(closes) / closes[:-1]
    volatility    = float(np.std(daily_returns))

    peak         = np.maximum.accumulate(closes)
    drawdowns    = (closes - peak) / peak
    max_drawdown = float(drawdowns.min())

    COMPANY_RESULTS[ticker]['volatility']   = volatility
    COMPANY_RESULTS[ticker]['max_drawdown'] = max_drawdown
    COMPANY_RESULTS[ticker]['risk_score']   = volatility + abs(max_drawdown)

print('risk scores computed')


In [ ]:
#@title 4.2 — rank by risk
risky = sorted(COMPANY_RESULTS.items(), key=lambda x: x[1]['risk_score'], reverse=True)
print(f"{'ticker':<30} {'risk_score':>12} {'volatility':>12} {'max_drawdown':>14}")
print('-' * 70)
for ticker, r in risky:
    print(f"{ticker:<30} {r['risk_score']:>12.4f} {r['volatility']:>12.4f} {r['max_drawdown']:>14.4%}")


In [ ]:
#@title 4.2 — flag risky companies
# Exclude companies above RISK_PERCENTILE (set in task 4 settings).
# Percentile-based threshold adapts to the volatility of the full universe.
scores         = np.array([r['risk_score'] for r in COMPANY_RESULTS.values()])
RISK_THRESHOLD = np.percentile(scores, RISK_PERCENTILE)

risky_tickers = [t for t, r in COMPANY_RESULTS.items() if r['risk_score'] >  RISK_THRESHOLD]
safe_tickers  = [t for t, r in COMPANY_RESULTS.items() if r['risk_score'] <= RISK_THRESHOLD]

print(f'threshold (p{RISK_PERCENTILE}): {RISK_THRESHOLD:.4f}')
print(f'risky: {len(risky_tickers)}  |  safe: {len(safe_tickers)}')


## 4.3: portfolio composition

In [ ]:
#@title 4.3 — compose portfolio
# Candidates = profitable (positive return) AND safe (below risk threshold).
portfolio_tickers = [t for t in profitable_tickers if t in safe_tickers]
print(f'portfolio candidates: {len(portfolio_tickers)}')
print(portfolio_tickers)


In [ ]:
#@title 4.3 — equal-weight allocation (prudent)
# Each company receives weight 1/N. Best for investors who want diversification
# over concentrated bets on the model's highest-confidence names.
eq_weight       = 1 / len(portfolio_tickers)
equal_portfolio = {t: eq_weight for t in portfolio_tickers}

print(f"{'ticker':<30} {'weight':>10}")
print('-' * 42)
for t, w in equal_portfolio.items():
    print(f"{t:<30} {w:>10.4%}")


In [ ]:
#@title 4.3 — return-weighted allocation (aggressive)
# Weight each company proportionally to its predicted return.
# Concentrates capital in the highest-upside names.
returns          = np.array([COMPANY_RESULTS[t]['predicted_return'] for t in portfolio_tickers])
weights          = returns / returns.sum()
return_portfolio = dict(zip(portfolio_tickers, weights))

print(f"{'ticker':<30} {'weight':>10} {'predicted_return':>18}")
print('-' * 60)
for t, w in sorted(return_portfolio.items(), key=lambda x: x[1], reverse=True):
    print(f"{t:<30} {w:>10.4%} {COMPANY_RESULTS[t]['predicted_return']:>18.4%}")


In [ ]:
#@title 4.3 — risk-taking vs prudent split
# Aggressive: return-weighted, predicted return >= median.
# Prudent:    equal-weight, predicted return < median.
median_return = np.median([COMPANY_RESULTS[t]['predicted_return'] for t in portfolio_tickers])

aggressive_portfolio = {t: w for t, w in return_portfolio.items()
                        if COMPANY_RESULTS[t]['predicted_return'] >= median_return}
prudent_portfolio    = {t: eq_weight for t in portfolio_tickers
                        if COMPANY_RESULTS[t]['predicted_return'] <  median_return}

print(f'median predicted return: {median_return:.4%}')
print(f'aggressive ({len(aggressive_portfolio)} stocks): {list(aggressive_portfolio.keys())}')
print(f'prudent    ({len(prudent_portfolio)} stocks):    {list(prudent_portfolio.keys())}')


# 5: Industry standard for deployment and ease of use

## 5.1: model deployment & 5.2: SaaS

In [ ]:
#@title verify stock_api.py
# stock_api.py is committed to the repo and available after the setup cell clones it.
# This cell confirms it's present before starting the server.
api_path = BASE / 'stock_api.py'
assert api_path.exists(), (
    f"stock_api.py not found at {api_path}\n"
    "Make sure it's committed to the repo root."
)
print(f'OK  stock_api.py at {api_path}')


In [ ]:
#@title install API dependencies
# fastapi + uvicorn only. cloudflared handles the tunnel (no pyngrok needed).
!pip install fastapi uvicorn --quiet


In [ ]:
#@title launch server
# Starts uvicorn in a background thread so the notebook stays interactive.
# cwd=BASE ensures stock_api.py is importable and its relative paths resolve correctly.
def _run_server():
    proc = subprocess.Popen(
        ['uvicorn', 'stock_api:app', '--host', '0.0.0.0', '--port', '8000'],
        cwd=str(BASE),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    for line in proc.stdout:
        print(line.decode(), end='')

threading.Thread(target=_run_server, daemon=True).start()
time.sleep(5)
print('server running on http://localhost:8000')


In [ ]:
#@title launch cloudflare tunnel
# Exposes localhost:8000 publicly via Cloudflare. No account required.
# URL changes on every launch.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=open('/tmp/cf.log', 'w'), stderr=subprocess.STDOUT,
)
time.sleep(6)

public_url = re.search(r'https://\S+\.trycloudflare\.com', open('/tmp/cf.log').read()).group()
print(f'public URL: {public_url}')
print(f'  /        frontend')
print(f'  /health  liveness check')
print(f'  /tickers available models')


In [ ]:
#@title test prediction request
# Sends a test request to the live API. Run after the tunnel cell.
import requests as _req

_ticker = Path(vn_files[0]).stem
_raw    = pd.read_csv(f'{VN_DATA_DIR}/{_ticker}.csv', index_col=0)
_raw    = _raw.sort_values('TradingDate').reset_index(drop=True)
_df     = add_technical_indicators(_raw)
_window = _df[VN_FEATURES].values[-VN_WINDOW_SIZE:].tolist()

_resp = _req.post(
    f'{public_url}/predict',
    json={'ticker': _ticker, 'window': _window, 'close_idx': VN_CLOSE_IDX},
)
print(json.dumps(_resp.json(), indent=2))
